In [ ]:
# Pooled Cumulative Incidence Curve – Optum Dataset

# This notebook generates a cumulative incidence curve for a cohort of people with HFpEF initating oral semaglutide vs sitagliptin.
# It uses overlap weights and plots cumulative incidence of the composite outcome: heart failure hospitalization or all-cause mortality.

# Required Inputs
## `pooled_weighted_dataset.csv` containing:
## `time`: follow-up time
## `event`: binary outcome indicator
## `treatment_group`: exposure (e.g. oral semaglutide vs sitagliptin)
## `weights`: overlap weights

In [1]:
import pandas as pd
import numpy as np
import scipy
import lifelines
import pandas
from lifelines import KaplanMeierFitter
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

# Load datasets
merged_optum_data = pd.read_csv("path/to/cohort_file.csv")
merged_optum_data_key = pd.read_csv("path/to/cohort_file.csv")

# Create and apply a column renaming mapping for merged_optum_data
cohort_mapping = dict(zip(merged_optum_data_key["Column Name"], merged_optum_data_key["Description"]))
cohort_mapping.pop('PID', None)  # Ensure 'PID' remains unchanged
trimmed_cohort_mapping = {col: desc.split(" - ")[-1] for col, desc in cohort_mapping.items()}
merged_optum_data.rename(columns=trimmed_cohort_mapping, inplace=True)

# Calculate overlap weights
merged_optum_data['overlap_weight'] = merged_optum_data['Propensity Score']
merged_optum_data.loc[merged_optum_data['Exposure Group'] == 'E', 'overlap_weight'] = 1 - merged_optum_data['Propensity Score']

# Inspect the final dataset
merged_optum_data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from matplotlib import font_manager as fm

# Step 1: Prepare data
merged_optum_data["Event"] = merged_optum_data["HHF or all-cause mortality"].astype(int)
merged_optum_data["Follow Up Time"] = np.minimum(merged_optum_data["Follow Up Time"], 365)
merged_optum_data["Exposure Binary"] = merged_optum_data["Exposure Group"].map({"E": 1, "R": 0})

# Step 2: Normalize overlap weights to ESS
ess = (merged_optum_data["overlap_weight"].sum() ** 2) / (merged_optum_data["overlap_weight"] ** 2).sum()
merged_optum_data["overlap_weight"] *= ess / merged_optum_data["overlap_weight"].sum()

# Step 3: Fit KM curves
km_sema = KaplanMeierFitter()
km_sita = KaplanMeierFitter()

km_sema.fit(
    durations=merged_optum_data.loc[merged_optum_data["Exposure Binary"] == 1, "Follow Up Time"],
    event_observed=merged_optum_data.loc[merged_optum_data["Exposure Binary"] == 1, "Event"],
    weights=merged_optum_data.loc[merged_optum_data["Exposure Binary"] == 1, "overlap_weight"],
    label="Oral semaglutide"
)

km_sita.fit(
    durations=merged_optum_data.loc[merged_optum_data["Exposure Binary"] == 0, "Follow Up Time"],
    event_observed=merged_optum_data.loc[merged_optum_data["Exposure Binary"] == 0, "Event"],
    weights=merged_optum_data.loc[merged_optum_data["Exposure Binary"] == 0, "overlap_weight"],
    label="Sitagliptin"
)

# Define colors
sema_red = "#a42b44"      # Oral semaglutide color from image
sita_gray = "#999999"     # Sitagliptin line
sita_gray_ci = "#aaaaaa"  # Sitagliptin CI band

fig, ax = plt.subplots(figsize=(6, 6))

# Oral semaglutide: use lifelines' built-in plot
km_sema.plot_cumulative_density(ax=ax, color=sema_red, label="Oral semaglutide", ci_show=True, lw=2)

# Sitagliptin: plot curve and CI manually
sita_curve = km_sita.cumulative_density_[km_sita.label]
ax.plot(
    sita_curve.index,
    sita_curve,
    label=km_sita.label,
    color=sita_gray,
    lw=2
)

# CI band for Sitagliptin
ci_lower = km_sita.confidence_interval_cumulative_density_[f"{km_sita.label}_lower_0.95"]
ci_upper = km_sita.confidence_interval_cumulative_density_[f"{km_sita.label}_upper_0.95"]
ax.fill_between(
    km_sita.cumulative_density_.index,
    ci_lower,
    ci_upper,
    color=sita_gray_ci,
    alpha=0.25
)

# X-axis (months)
month_days = np.array([0, 3, 6, 9, 12]) * 30
month_days[-1] = 365
ax.set_xlim(0, 365)
ax.set_xticks(month_days)
ax.set_xticklabels([f"{m}" for m in [0, 3, 6, 9, 12]])

# Y-axis (percent scale with grid)
ax.set_ylim(0.00, 0.10)
ax.set_yticks(np.arange(0.00, 0.11, 0.02))
ax.set_yticklabels([f"{int(y*100)}" for y in np.arange(0.00, 0.11, 0.02)])

# Axis labels
ax.set_xlabel("Follow-up, mo")
ax.set_ylabel("Cumulative incidence, %")

# Legend
legend = ax.legend(loc='upper left', bbox_to_anchor=(0.02, 0.98), frameon=True,
                   handlelength=2, handletextpad=1, borderpad=0.2, labelspacing=0.4)
legend.get_frame().set_boxstyle("square")
legend.get_frame().set_linewidth(1.0)

# Grid and layout
ax.grid(True, which='major', axis='y', linestyle='-', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Extract point estimates
# Semaglutide
sema_surv = km_sema.predict(365)
sema_ci = km_sema.confidence_interval_.loc[365]
sema_label = km_sema.label
sema_lower_surv = sema_ci[f"{sema_label}_lower_0.95"]
sema_upper_surv = sema_ci[f"{sema_label}_upper_0.95"]
sema_cum = 1 - sema_surv
sema_lower = 1 - sema_upper_surv
sema_upper = 1 - sema_lower_surv

# Sitagliptin
sita_surv = km_sita.predict(365)
sita_ci = km_sita.confidence_interval_.loc[365]
sita_label = km_sita.label
sita_lower_surv = sita_ci[f"{sita_label}_lower_0.95"]
sita_upper_surv = sita_ci[f"{sita_label}_upper_0.95"]
sita_cum = 1 - sita_surv
sita_lower = 1 - sita_upper_surv
sita_upper = 1 - sita_lower_surv

# Calculate ARD
def bootstrap_ard(df, n_iterations=1000, seed=42):
    np.random.seed(seed)
    boot_ard = []

    for _ in range(n_iterations):
        boot_df = df.sample(n=len(df), replace=True)

        # Ensure follow-up is capped and binary event is clean
        boot_df["Follow Up Time"] = np.minimum(boot_df["Follow Up Time"], 365)
        boot_df["Event"] = boot_df["HHF or all-cause mortality"].astype(int)
        boot_df["Exposure Binary"] = boot_df["Exposure Group"].map({"E": 1, "R": 0})

        km = KaplanMeierFitter()
        results = {}

        for group_name, group_val in [("Semaglutide", 1), ("Sitagliptin", 0)]:
            mask = boot_df["Exposure Binary"] == group_val
            km.fit(
                durations=boot_df.loc[mask, "Follow Up Time"],
                event_observed=boot_df.loc[mask, "Event"],
                weights=boot_df.loc[mask, "overlap_weight"]
            )
            surv_prob = km.predict(365)
            results[group_name] = 1 - surv_prob

        boot_ard.append(results["Semaglutide"] - results["Sitagliptin"])

    return np.array(boot_ard)

boot_ard_dist = bootstrap_ard(merged_optum_data, n_iterations=1000)
ard_point = sema_cum - sita_cum
ard_lower = np.percentile(boot_ard_dist, 2.5)
ard_upper = np.percentile(boot_ard_dist, 97.5)

# Create summary DataFrame with fixed 2-decimal formatting
summary_df = pd.DataFrame({
    "Comparison": [
        f"{sema_label}",
        f"{sita_label}",
        f"{sema_label} vs {sita_label} (ARD)"
    ],
    "Estimate (%)": [
        f"{sema_cum * 100:.2f}",
        f"{sita_cum * 100:.2f}",
        f"{ard_point * 100:.2f}"
    ],
    "95% CI (%)": [
        f"{sema_lower * 100:.2f}–{sema_upper * 100:.2f}",
        f"{sita_lower * 100:.2f}–{sita_upper * 100:.2f}",
        f"{ard_lower * 100:.2f}–{ard_upper * 100:.2f}"
    ]
})

summary_df